# Esse resultado é confiável?

**Nível 2 — Intermediário** · Trilha Wizz Lab

> Uma estimativa não é um fato. Todo número do nível 1 tem um intervalo em volta.


Três ideias novas: a **forma** da distribuição, a **incerteza** da estimativa e o
**custo**.


---

In [ ]:
%matplotlib inline

# No Colab, instala o pacote direto do GitHub. Localmente, não faz nada.
import importlib.util, subprocess, sys

if importlib.util.find_spec("wizzlab") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/Gustavofthiesen/wizz-lab.git"], check=True)

from wizzlab import brand, data, metrics, scorecard
from wizzlab.theme import aplicar_tema
from wizzlab import charts

aplicar_tema()
print("Wizz Lab pronto · paleta:", brand.SERIE_PRINCIPAL, brand.SERIE_COMPARACAO)

In [ ]:
est = data.gerar_estrategia(semente=42)
r = est.trades.r_multiple.values
print(f"Expectancy observada: {metrics.trade.expectancy(r):+.3f}R em {len(r)} trades")

## 1. O edge é um intervalo, não um ponto

Aquele `+0,08R por trade` do nível 1 é uma estimativa feita sobre uma amostra finita.
Reamostrando os próprios trades com reposição (*bootstrap*), vemos que faixa de valores
os dados realmente suportam.

In [ ]:
fig, ax = charts.intervalo_bootstrap(r, fonte="Dados simulados · wizz-lab")
fig

In [ ]:
lo, hi = metrics.evidencia.bootstrap_ci(r)
print(f"Expectancy observada .. {metrics.trade.expectancy(r):+.3f}R")
print(f"IC 95% ................ [{lo:+.3f}R ; {hi:+.3f}R]")
print(f"P(edge > 0) ........... {metrics.evidencia.prob_expectancy_positiva(r):.1%}")
print(f"t-stat ................ {metrics.evidencia.t_stat(r):.2f}")

> **Aqui está a lição central do notebook.** Esta estratégia tem, por construção, uma
> expectancy verdadeira de aproximadamente **+0,22R** — um edge real e economicamente
> relevante. Ainda assim, com 420 trades, o intervalo de confiança **cruza o zero**.
>
> Não é defeito do gerador. É o tamanho de amostra que a maioria dos backtests tem.

Aumente a amostra e veja o mesmo processo virar significativo:

In [ ]:
grande = data.gerar_estrategia(n_trades=3000, semente=42)
rg = grande.trades.r_multiple.values
lo2, hi2 = metrics.evidencia.bootstrap_ci(rg)

print(f"420 trades  → t = {metrics.evidencia.t_stat(r):5.2f}   IC [{lo:+.3f} ; {hi:+.3f}]")
print(f"3000 trades → t = {metrics.evidencia.t_stat(rg):5.2f}   IC [{lo2:+.3f} ; {hi2:+.3f}]")

Mesmo processo gerador. Mesma expectancy verdadeira. **A diferença é só quanta evidência
existe** — e é por isso que "a estratégia funciona" e "eu consigo demonstrar que a
estratégia funciona" são afirmações diferentes.

## 2. A forma da distribuição

Média e desvio não descrevem uma distribuição assimétrica. Faltam a assimetria e as
caudas.

In [ ]:
import pandas as pd
pd.Series({
    "Skewness": metrics.trade.skewness(r),
    "Excess kurtosis": metrics.trade.excess_kurtosis(r),
    "Tail ratio": metrics.trade.tail_ratio(r),
    "IQR": metrics.trade.iqr(r),
    "MAD": metrics.trade.mad(r),
}).round(3)

Skew positivo = cauda direita = o perfil de quem aceita muitas perdas pequenas em troca
de poucos ganhos grandes.

O padrão **perigoso** é o inverso: Sharpe alto com skew fortemente negativo. Significa
ganhos pequenos e constantes com uma perda enorme que ainda não aconteceu no período
medido.

## 3. Risco de cauda: onde a cauda começa e o que há dentro dela

In [ ]:
pd.Series({
    "VaR 5% (diário)": metrics.risco.var_historico(est.diario),
    "Expected Shortfall 5%": metrics.risco.expected_shortfall(est.diario),
    "Maior sequência de perdas": metrics.risco.max_streak(r),
    "Kelly fraction": metrics.risco.kelly_fraction(r),
    "Risk of ruin (perder 50%)": metrics.risco.risk_of_ruin(r, fracao_por_trade=0.01),
}).round(4)

**VaR e Expected Shortfall formam um par que ensina sozinho.** O VaR diz onde a cauda
ruim começa; o ES diz qual é a perda média *dentro* dela. Quem reporta só o primeiro está
escondendo o segundo.

Sobre Kelly: é um teto teórico, não uma recomendação. Ele supõe que a distribuição
estimada está certa — e ela nunca está. Kelly cheio sobre parâmetros estimados produz
drawdowns intoleráveis.

## 4. Monte Carlo: o histórico foi só um dos caminhos possíveis

In [ ]:
fig, ax = charts.monte_carlo(r, fonte="Dados simulados · wizz-lab")
fig

In [ ]:
mdd_obs = metrics.risco.max_drawdown(est.diario)
fig, ax = charts.distribuicao_mdd(r, mdd_observado=mdd_obs,
                                  fonte="Dados simulados · wizz-lab")
fig

In [ ]:
metrics.risco.monte_carlo_mdd(r).round(3)

> **A figura que muda decisão de sizing.** O drawdown observado fica bem à direita da
> distribuição — ou seja, o histórico foi sortudo. Quem dimensiona posição pelo MDD
> observado está dimensionando pela sorte.
>
> O número que importa para o sizing é o **P95**, não o observado.

## 5. Custo: o assassino silencioso

Lembra da margem de 3 pontos percentuais sobre o break-even, do nível 1?

In [ ]:
metrics.execucao.slippage_sensitivity(r, custos=(0.0, 0.05, 0.10, 0.20, 0.40)).round(3)

In [ ]:
print(f"Custo que zera o edge ..... {metrics.execucao.break_even_cost(r):.3f}R")
print(f"Margem sobre custo de 0,05R {metrics.execucao.cost_safety_margin(r, 0.05):.2f}x")

**O que observar não é o nível de cada linha — é a inclinação.** Uma estratégia cujo edge
cai pela metade com 0,05R de custo extra não existe fora do backtest.

Abaixo de 2x de margem eu não dormiria tranquilo: qualquer alargamento de spread ou
mudança de corretagem come a folga.

## 6. Fora da amostra, pela primeira vez

Todo número até aqui foi calculado sobre os mesmos dados que escolheram as regras. Hora de
separar.

In [ ]:
print(f"Retenção OOS/IS ........... {metrics.generalizacao.oos_is_ratio(r):.2f}")
print(f"Walk-forward efficiency ... {metrics.generalizacao.walk_forward_efficiency(r):.2f}")
print(f"Janelas OOS positivas ..... {metrics.generalizacao.positive_oos_windows(r):.0%}")
print()
metrics.generalizacao.walk_forward(r, n_janelas=6).round(4)

Nunca divida uma série temporal aleatoriamente: isso vaza o futuro para o treino. A
divisão é sempre **na ordem do tempo**.

## 7. Robustez: o ótimo é um platô ou um pico?

In [ ]:
import numpy as np
# Superfície simulada: um platô largo e suave é o que se quer ver.
eixo_a, eixo_b = np.arange(24), np.arange(20)
superficie = np.add.outer(np.exp(-np.linspace(-2, 2, 20) ** 2 / 2.0),
                          np.exp(-np.linspace(-2, 2, 24) ** 2 / 1.4))
fig, ax = charts.superficie_parametros(superficie, eixo_a, eixo_b,
                                       rotulo_x="janela do indicador",
                                       rotulo_y="limiar de entrada",
                                       fonte="Superfície simulada · wizz-lab")
fig

In [ ]:
resultados = pd.Series(superficie.mean(axis=0), index=eixo_a)
print(metrics.generalizacao.parameter_stability(resultados).round(3).to_string())
print()
print("Largura do platô (>=90% do ótimo):",
      f"{metrics.generalizacao.parameter_plateau_width(resultados):.0%}")

**Regiões largas e suaves convencem; picos estreitos, não.** Um ótimo isolado cercado de
resultados ruins é quase sempre ruído que foi escolhido depois do fato.

## 8. Concentração: quantas apostas existem de verdade?

In [ ]:
fig, ax = charts.concentracao_pnl(r, fonte="Dados simulados · wizz-lab")
fig

In [ ]:
print(metrics.sinal.pnl_concentration(r).round(3).to_string())
print()
print(metrics.sinal.remove_best_trades(r, n=5).to_string())
print()
print("Nº efetivo de ativos (breadth):",
      round(metrics.sinal.breadth(r, est.trades.ativo), 2))

Se remover cinco trades de 420 zera o edge, a estratégia depende de **eventos**, não de um
processo. E dez ativos dos quais um responde por 80% do lucro não são dez apostas.

---

## O que levar deste nível

1. Toda métrica é uma estimativa. Reporte o intervalo, não o ponto.
2. Um edge real pode não ser demonstrável com a amostra que você tem.
3. VaR diz onde a cauda começa; ES diz o que há dentro dela.
4. Dimensione pelo MDD simulado P95, não pelo observado.
5. Na sensibilidade a custo, olhe a inclinação, não o nível.
6. Platô largo convence; pico estreito não.

**No nível 3:** tudo isso ainda supõe que você testou uma estratégia. E se você testou
duzentas e mostrou a melhor?

---

### Bloco de transparência

**Natureza:** educacional · **Dados:** simulados e reprodutíveis por semente ·
**Código:** aberto em [wizz-lab](https://github.com/Gustavofthiesen/wizz-lab)

Este material apresenta um processo de estudo, com finalidade educacional. Não
constitui recomendação individualizada, oferta ou promessa de retorno. Premissas podem
estar erradas e resultados passados não garantem resultados futuros.